# LeetCode #103 — Binary Tree Zigzag Level Order Traversal: BFS tầng, đổi chiều xen kẽ

**Mức độ:** Medium · **Chủ đề:** Tree, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def zigzagLevelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
```

Bài này là #102 với một thay đổi nhỏ: tầng chẵn đọc **trái → phải**, tầng lẻ đọc **phải → trái**. Mục tiêu của notebook:

1. Tái sử dụng khung BFS theo tầng của #102.
2. Ba cách đổi chiều: **đảo list**, **`deque` hai đầu** (`appendleft`), và **hai stack**.
3. Hiểu vì sao **không** nên đổi thứ tự **thêm con** vào hàng đợi.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, trả về giá trị theo **thứ tự zigzag**: tầng 0 từ trái sang phải, tầng 1 từ phải sang trái, tầng 2 lại từ trái sang phải, … xen kẽ.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2000]`.
- `-100 <= Node.val <= 100`

---

## Ví dụ

```text
    3
   / \
  9  20          ->  [[3], [20, 9], [15, 7]]
    /  \
   15   7

root = [1]  ->  [[1]]
root = []   ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Hàng đợi (queue) và `collections.deque`
3. Duyệt theo chiều rộng (BFS — Breadth-First Search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Hàng đợi (queue) và `collections.deque`

### Hàng đợi là gì?

Hàng đợi là cấu trúc **vào trước, ra trước** (FIFO — First In, First Out), giống xếp hàng mua vé: người đến trước được phục vụ trước. Thêm ở **cuối**, lấy ra ở **đầu**.

```text
enqueue 1, 2, 3:     đầu → [1, 2, 3] ← cuối
dequeue → 1:         đầu → [2, 3]
enqueue 4:           đầu → [2, 3, 4]
```

### Vì sao không dùng `list.pop(0)`?

`list` lưu liền nhau; bỏ phần tử đầu buộc **dời** mọi phần tử còn lại sang trái → `O(n)`. `collections.deque` (hàng đợi hai đầu) thêm/bỏ ở **cả hai đầu** trong `O(1)`.

| Thao tác | `deque` | `list` |
|---|---|---|
| thêm cuối `append` | `O(1)` | `O(1)` |
| bỏ đầu `popleft` / `pop(0)` | `O(1)` | `O(n)` |
| thêm đầu `appendleft` / `insert(0, x)` | `O(1)` | `O(n)` |

In [6]:
from collections import deque
import time as _time_queue

q = deque()
for x in [1, 2, 3]:
    q.append(x)                  # vào cuối
print("hàng đợi:", q)
print("popleft:", q.popleft(), "->", q)   # ra đầu
q.appendleft(0)
print("appendleft(0):", q, "| xem đầu:", q[0], "| xem cuối:", q[-1], "| len:", len(q))

_n = 50_000
_t0 = _time_queue.perf_counter()
_l = list(range(_n))
while _l:
    _l.pop(0)
_t1 = _time_queue.perf_counter()
_d = deque(range(_n))
while _d:
    _d.popleft()
_t2 = _time_queue.perf_counter()
print(f"bỏ đầu {_n} lần: list.pop(0) {_t1 - _t0:.3f}s | deque.popleft {_t2 - _t1:.3f}s")

hàng đợi: deque([1, 2, 3])
popleft: 1 -> deque([2, 3])
appendleft(0): deque([0, 2, 3]) | xem đầu: 0 | xem cuối: 3 | len: 3


bỏ đầu 50000 lần: list.pop(0) 0.347s | deque.popleft 0.003s


### Khi nào dùng hàng đợi?

- **BFS (duyệt theo chiều rộng)**: xử lý các đỉnh theo thứ tự được phát hiện — gần trước, xa sau.
- Mô phỏng hàng chờ, xử lý tác vụ theo lượt.
- `deque` cũng làm được ngăn xếp (`append` + `pop`) và **cửa sổ trượt** (bỏ phần tử cũ ở đầu).

Lỗi hay gặp: `q.pop()` (bỏ **cuối**) thay vì `q.popleft()` → vô tình biến hàng đợi thành ngăn xếp, BFS thành DFS.

## 2.3 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [7]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [8]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [9]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.4 Áp dụng vào bài này

**BFS theo tầng** như #102; khác biệt duy nhất là tầng lẻ ghi giá trị theo chiều ngược lại — dùng `deque.appendleft` (thêm vào **đầu** trong `O(1)`) hoặc đảo list của tầng.

# 3. Nền tảng

In [10]:
from typing import List, Optional
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def level_order(root: Optional[TreeNode]) -> List[List[int]]:
    """BFS theo tầng (#102)."""
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


# 4. Cách 1: BFS bình thường, đảo các tầng lẻ

Đơn giản nhất: lấy kết quả của #102, đảo ngược tầng có chỉ số lẻ. Đảo một list độ dài `k` tốn `O(k)` → tổng vẫn `O(n)`.

In [11]:
def zigzag_reverse(root: Optional[TreeNode]) -> List[List[int]]:
    levels = level_order(root)
    return [lvl if d % 2 == 0 else lvl[::-1] for d, lvl in enumerate(levels)]

print(zigzag_reverse(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [20, 9], [15, 7]]


# 5. Cách 2: `deque` cho mỗi tầng — `append` hoặc `appendleft`

Vẫn duyệt hàng đợi **trái → phải** như bình thường, nhưng khi **ghi** giá trị vào tầng hiện tại:

- Tầng trái → phải: `level.append(val)`
- Tầng phải → trái: `level.appendleft(val)` — giá trị duyệt sau đứng **trước**.

In [12]:
def zigzag_deque(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    left_to_right = True
    while q:
        level = deque()
        for _ in range(len(q)):
            node = q.popleft()
            if left_to_right:
                level.append(node.val)
            else:
                level.appendleft(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(list(level))
        left_to_right = not left_to_right
    return result

print(zigzag_deque(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [20, 9], [15, 7]]


**Điểm mấu chốt:** thứ tự **duyệt** và thêm con vào hàng đợi **không đổi** (luôn trái trước phải). Ta chỉ đổi cách **ghi** kết quả. Nhờ vậy tầng sau vẫn được duyệt đúng thứ tự trái → phải.

# 6. Mô phỏng

In [13]:
def trace_zigzag(values: list) -> List[List[int]]:
    root = build_tree(values)
    result, q = [], deque([root]) if root else deque()
    left_to_right, depth = True, 0
    while q:
        level = deque()
        print(f"Tầng {depth} ({'trái → phải' if left_to_right else 'phải → trái'}), hàng đợi = {[n.val for n in q]}")
        for _ in range(len(q)):
            node = q.popleft()
            if left_to_right:
                level.append(node.val)
                how = "append    "
            else:
                level.appendleft(node.val)
                how = "appendleft"
            print(f"   {how} {node.val} -> level = {list(level)}")
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(list(level))
        left_to_right = not left_to_right
        depth += 1
    print("Kết quả:", result)
    return result

_ = trace_zigzag([1, 2, 3, 4, 5, 6, 7, 8, 9])

Tầng 0 (trái → phải), hàng đợi = [1]
   append     1 -> level = [1]
Tầng 1 (phải → trái), hàng đợi = [2, 3]
   appendleft 2 -> level = [2]
   appendleft 3 -> level = [3, 2]
Tầng 2 (trái → phải), hàng đợi = [4, 5, 6, 7]
   append     4 -> level = [4]
   append     5 -> level = [4, 5]
   append     6 -> level = [4, 5, 6]
   append     7 -> level = [4, 5, 6, 7]
Tầng 3 (phải → trái), hàng đợi = [8, 9]
   appendleft 8 -> level = [8]
   appendleft 9 -> level = [9, 8]
Kết quả: [[1], [3, 2], [4, 5, 6, 7], [9, 8]]


# 7. Cách 3: Hai stack

Dùng hai stack luân phiên. Lấy nút ra từ stack hiện tại (đỉnh = cuối), đẩy con vào stack kia:

- Tầng trái → phải: đẩy **trái rồi phải** → ở tầng sau phải được lấy ra trước.
- Tầng phải → trái: đẩy **phải rồi trái** → ở tầng sau trái được lấy ra trước.

Cách này **đổi thứ tự thêm con**, nhưng nhờ tính LIFO của stack nên vẫn đúng.

In [14]:
def zigzag_two_stacks(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result = []
    cur, nxt = [root], []
    left_to_right = True
    while cur:
        level = []
        while cur:
            node = cur.pop()
            level.append(node.val)
            first, second = (node.left, node.right) if left_to_right else (node.right, node.left)
            if first:
                nxt.append(first)
            if second:
                nxt.append(second)
        result.append(level)
        cur, nxt = nxt, []
        left_to_right = not left_to_right
    return result

print(zigzag_two_stacks(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])))

[[1], [3, 2], [4, 5, 6, 7], [9, 8]]


# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def zigzagLevelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:
            return []
        result, q = [], deque([root])
        left_to_right = True
        while q:
            level = deque()
            for _ in range(len(q)):
                node = q.popleft()
                if left_to_right:
                    level.append(node.val)
                else:
                    level.appendleft(node.val)
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
            result.append(list(level))
            left_to_right = not left_to_right
        return result

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def random_tree(n):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(-9, 9), random_tree(k), random_tree(n - 1 - k))


sol = Solution()
check("Ví dụ 1", sol.zigzagLevelOrder(build_tree([3, 9, 20, None, None, 15, 7])), [[3], [20, 9], [15, 7]])
check("Ví dụ 2", sol.zigzagLevelOrder(build_tree([1])), [[1]])
check("Ví dụ 3", sol.zigzagLevelOrder(build_tree([])), [])
check("Cây 4 tầng", sol.zigzagLevelOrder(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])), [[1], [3, 2], [4, 5, 6, 7], [9, 8]])

random.seed(103)
for _ in range(3000):
    root = random_tree(random.randint(0, 15))
    expected = zigzag_reverse(root)
    assert sol.zigzagLevelOrder(root) == expected == zigzag_two_stacks(root) == zigzag_deque(root)
print("3000 cây ngẫu nhiên: 3 cách khớp nhau")

Ví dụ 1          PASS  [[3], [20, 9], [15, 7]]
Ví dụ 2          PASS  [[1]]
Ví dụ 3          PASS  []
Cây 4 tầng       PASS  [[1], [3, 2], [4, 5, 6, 7], [9, 8]]


3000 cây ngẫu nhiên: 3 cách khớp nhau

# 9. Vì sao đúng? Độ phức tạp?

- BFS theo tầng (bất biến của #102) cho các nút mỗi tầng theo thứ tự trái → phải.
- Tầng chẵn ghi theo thứ tự đó; tầng lẻ dùng `appendleft` nên thứ tự bị **đảo** — đúng yêu cầu.

```text
Time:  O(n)
Space: O(w) — w là độ rộng lớn nhất của một tầng
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Đổi thứ tự **thêm con** vào **hàng đợi** (FIFO)

Ý tưởng sai: "ở tầng lẻ thì thêm con phải trước". Với hàng đợi, điều này làm hỏng thứ tự của **tầng sau nữa**.

In [17]:
def wrong_swap_children_queue(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    left_to_right = True
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            kids = (node.left, node.right) if not left_to_right else (node.right, node.left)   # SAI
            for k in kids:
                if k:
                    q.append(k)
        result.append(level)
        left_to_right = not left_to_right
    return result

print("Sai :", wrong_swap_children_queue(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])))
print("Đúng:", sol.zigzagLevelOrder(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])))

Sai : [[1], [3, 2], [6, 7, 4, 5], [9, 8]]
Đúng: [[1], [3, 2], [4, 5, 6, 7], [9, 8]]


Tầng 1 đúng (`[3, 2]`), nhưng tầng 2 thành `[6, 7, 4, 5]` thay vì `[4, 5, 6, 7]`: vì hàng đợi của tầng 1 là `3, 2` (đã đảo), các con của `3` (`6, 7`) bị xếp **trước** các con của `2` (`4, 5`). Đảo thứ tự thêm con chỉ đảo **trong từng cặp anh em**, không đảo được cả tầng.

## Lỗi 2: Quên đổi cờ `left_to_right` sau mỗi tầng

Kết quả giống hệt #102.

## Lỗi 3: Đảo bằng `level.reverse()` nhưng đặt sai chỗ

Nếu `reverse()` trước khi duyệt xong tầng (trong vòng `for`), mỗi lần đảo một phần → thứ tự lộn xộn. Đảo **sau** vòng `for`.

## Lỗi 4: Hai stack nhưng không đổi thứ tự đẩy con

Với stack (LIFO), nếu luôn đẩy trái rồi phải, mọi tầng sau tầng 0 đều bị đọc phải → trái.

In [18]:
def wrong_two_stacks_same_order(root: Optional[TreeNode]) -> List[List[int]]:
    result, cur, nxt = [], [root], []
    while cur:
        level = []
        while cur:
            node = cur.pop()
            level.append(node.val)
            for k in (node.left, node.right):           # SAI: luôn cùng thứ tự
                if k:
                    nxt.append(k)
        result.append(level)
        cur, nxt = nxt, []
    return result

print("Sai :", wrong_two_stacks_same_order(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])))

Sai : [[1], [3, 2], [5, 4, 7, 6], [9, 8]]


# 11. Mẹo nhận diện pattern

Biến thể của **BFS theo tầng**: chỉ thay đổi cách **ghi** kết quả mỗi tầng (đảo, lấy cuối, lấy max, trung bình…), giữ nguyên cách **duyệt**.

Bài liên quan: **#102**, **#107 Level Order Traversal II** (đảo thứ tự các tầng), **#199 Right Side View**, **#637 Average of Levels in Binary Tree**, **#2415 Reverse Odd Levels of Binary Tree**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Zigzag của cây `[1, 2, 3, 4, None, None, 5, 6, None, None, 7]`.

## Bài 2 — Điền chỗ trống

In [19]:
def exercise_2_zigzag(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    left_to_right = True
    while q:
        level = deque()
        for _ in range(len(q)):
            node = q.popleft()
            # TODO: ghi node.val theo chiều; thêm con (luôn trái trước phải)
            pass
        result.append(list(level))
        # TODO: đổi chiều
    return result


# assert exercise_2_zigzag(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [20, 9], [15, 7]]
# assert exercise_2_zigzag(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])) == [[1], [3, 2], [4, 5, 6, 7], [9, 8]]

## Bài 3 — Biến thể: Binary Tree Level Order Traversal II (LeetCode #107)

Trả các tầng từ **dưới lên** (tầng sâu nhất trước), mỗi tầng vẫn trái → phải.

```python
level_order_bottom([3, 9, 20, None, None, 15, 7]) -> [[15, 7], [9, 20], [3]]
```

In [20]:
def exercise_3_level_order_bottom(root: Optional[TreeNode]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_3_level_order_bottom(build_tree([3, 9, 20, None, None, 15, 7])) == [[15, 7], [9, 20], [3]]
# assert exercise_3_level_order_bottom(build_tree([])) == []

## Bài 4 — Biến thể: Average of Levels (LeetCode #637)

Trung bình cộng các giá trị mỗi tầng.

```python
average_of_levels([3, 9, 20, None, None, 15, 7]) -> [3.0, 14.5, 11.0]
```

In [21]:
def exercise_4_average_of_levels(root: Optional[TreeNode]) -> List[float]:
    # TODO
    pass


# assert exercise_4_average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
# assert exercise_4_average_of_levels(build_tree([3, 9, 20, 15, 7])) == [3.0, 14.5, 11.0]

## Bài 5 — Mini challenge: Reverse Odd Levels of Binary Tree (LeetCode #2415)

Cây nhị phân **hoàn hảo** (mọi tầng đầy đủ). Đảo ngược **giá trị** các nút ở mọi tầng lẻ, tại chỗ.

```python
reverse_odd_levels([2, 3, 5, 8, 13, 21, 34])  -> [2, 5, 3, 8, 13, 21, 34]
reverse_odd_levels([0, 1, 2, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]) -> [0, 2, 1, 0, 0, 0, 0, 2, 2, 2, 2, 1, 1, 1, 1]
```

Gợi ý nhanh: BFS lấy list **nút** của mỗi tầng; ở tầng lẻ, đổi giá trị hai đầu tiến dần vào giữa.

In [22]:
def exercise_5_reverse_odd_levels(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# assert level_order(exercise_5_reverse_odd_levels(build_tree([2, 3, 5, 8, 13, 21, 34]))) == [[2], [5, 3], [8, 13, 21, 34]]
# assert level_order(exercise_5_reverse_odd_levels(build_tree([0, 1, 2, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]))) == [[0], [2, 1], [0, 0, 0, 0], [2, 2, 2, 2, 1, 1, 1, 1]]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_zigzag` sau khi tự làm.

**Bài 3**
- Mức 1: `level_order(root)[::-1]`, hoặc `appendleft` từng tầng vào một `deque` kết quả.

**Bài 4**
- Mức 1: trong khung BFS, cộng tổng và chia cho `size` của tầng.

**Bài 5**
- Mức 1: giữ `level_nodes` (list nút) cho mỗi tầng; `depth % 2 == 1` thì `i, j = 0, len - 1` và đổi `.val`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
_ = trace_zigzag([1, 2, 3, 4, None, None, 5, 6, None, None, 7])

Tầng 0 (trái → phải), hàng đợi = [1]
   append     1 -> level = [1]
Tầng 1 (phải → trái), hàng đợi = [2, 3]
   appendleft 2 -> level = [2]
   appendleft 3 -> level = [3, 2]
Tầng 2 (trái → phải), hàng đợi = [4, 5]
   append     4 -> level = [4]
   append     5 -> level = [4, 5]
Tầng 3 (phải → trái), hàng đợi = [6, 7]
   appendleft 6 -> level = [6]
   appendleft 7 -> level = [7, 6]
Kết quả: [[1], [3, 2], [4, 5], [7, 6]]


## Đáp án Bài 2

In [24]:
def answer_2_zigzag(root: Optional[TreeNode]) -> List[List[int]]:
    return zigzag_deque(root)

assert answer_2_zigzag(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [20, 9], [15, 7]]
assert answer_2_zigzag(build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])) == [[1], [3, 2], [4, 5, 6, 7], [9, 8]]
print("Bài 2 PASS (giống zigzag_deque ở mục 5)")

Bài 2 PASS (giống zigzag_deque ở mục 5)


## Đáp án Bài 3

In [25]:
def answer_3_level_order_bottom(root: Optional[TreeNode]) -> List[List[int]]:
    return level_order(root)[::-1]

assert answer_3_level_order_bottom(build_tree([3, 9, 20, None, None, 15, 7])) == [[15, 7], [9, 20], [3]]
assert answer_3_level_order_bottom(build_tree([])) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_average_of_levels(root: Optional[TreeNode]) -> List[float]:
    return [sum(lvl) / len(lvl) for lvl in level_order(root)]

assert answer_4_average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
assert answer_4_average_of_levels(build_tree([3, 9, 20, 15, 7])) == [3.0, 14.5, 11.0]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_reverse_odd_levels(root: Optional[TreeNode]) -> Optional[TreeNode]:
    level_nodes, depth = [root] if root else [], 0
    while level_nodes:
        if depth % 2 == 1:
            i, j = 0, len(level_nodes) - 1
            while i < j:
                level_nodes[i].val, level_nodes[j].val = level_nodes[j].val, level_nodes[i].val
                i += 1
                j -= 1
        level_nodes = [c for n in level_nodes for c in (n.left, n.right) if c]
        depth += 1
    return root

assert level_order(answer_5_reverse_odd_levels(build_tree([2, 3, 5, 8, 13, 21, 34]))) == [[2], [5, 3], [8, 13, 21, 34]]
assert level_order(answer_5_reverse_odd_levels(build_tree([0, 1, 2, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]))) == [[0], [2, 1], [0, 0, 0, 0], [2, 2, 2, 2, 1, 1, 1, 1]]
print("Bài 5 PASS")

Bài 5 PASS


Ở đây ta xây danh sách nút tầng sau bằng list comprehension thay vì `deque` — cũng là BFS theo tầng, chỉ khác cách cài đặt.

# 15. Tổng kết

> BFS theo tầng như #102; duyệt và thêm con **luôn trái trước phải**; chỉ đổi cách **ghi**: `append` ở tầng chẵn, `appendleft` (hoặc đảo list) ở tầng lẻ. Đổi cờ sau mỗi tầng.

---

## Checklist trước khi nộp

- [ ] Không đổi thứ tự thêm con vào **hàng đợi**?
- [ ] Đổi cờ chiều sau **mỗi** tầng?
- [ ] `root = None` → `[]`?
- [ ] Nếu dùng hai stack: đổi thứ tự đẩy con theo tầng?